# Unsupervised Learning Project
## Module E3 – Fuzzy c-Means on the Main Representation

**Team (G4-P2):** José Nunes (73137) · João Lourenço (72904) · Leonor Afonso (73491)

**Course:** Unsupervised Learning · 2025/2026 · NOVA FCT, Departamento de Informática

---

**E3 Overview.**
This notebook applies **Fuzzy c-Means (FCM)** to the representation `R0` used in Tasks 1
and 2. FCM is a soft-clustering extension of K-Means: each booking receives a **membership
vector** `w_ik ∈ [0,1]` to every cluster instead of a hard label, with the constraint
`Σ_k w_ik = 1` for every observation `i`. The degree of fuzziness is governed by the
weighting exponent `m > 1`.

**Why this module?**
Task 1 reported a bootstrap stability of mean pairwise ARI = 0.644 at the chosen `K = 4` –
above the unstable threshold (0.60) but below the strictly stable threshold (0.75).
The hypothesis we test here is that this moderate stability reflects a **diffuse boundary**
between two of the four clusters. Fuzzy c-Means quantifies that hypothesis directly through
the membership matrix **W**.

**Notebook layout.**
1. Reproduce the R0 representation from Task 1.
2. Implement FCM (Alternating Optimization) from scratch (numpy) following slide notation.
3. Calibrate the fuzziness exponent `m` on R0.
4. Run the main FCM fit at `K = 4` and the calibrated `m`.
5. Report fuzzy validity indices (Partition Coefficient, Classification Entropy, Xie–Beni).
6. Quantify membership confidence per cluster and identify ambiguous bookings.
7. Compare the FCM hard partition with the K-Means partition via Adjusted Rand Index.
8. Profile FCM clusters in original units and append all runs to `experiments.csv`.


## 1. Environment, Imports and Reproducibility

Constants and project paths are **identical to Tasks 1 and 2** so that the FCM result is
computed on the same `X_R0` matrix.


In [ ]:
import os
import time
import hashlib
import warnings
from pathlib import Path
from itertools import combinations
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score,
)

# Reproducibility (identical to Tasks 1 and 2)
SEED            = 12345
SEEDS_STABILITY = [12345, 23456, 34567, 45678, 56789]
SUBSAMPLE_N     = 10_000
np.random.seed(SEED)

# Project layout (identical to Tasks 1 and 2)
def _find_project_root(markers=('requirements.txt', '.git', 'environment.yml')):
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if any((candidate / m).exists() for m in markers):
            return candidate
    return here

PROJECT_ROOT = _find_project_root()
DATA_DIR     = PROJECT_ROOT / 'data' / 'raw'
RESULTS_DIR  = PROJECT_ROOT / 'results'
FIG_DIR      = RESULTS_DIR / 'figures'
TBL_DIR      = RESULTS_DIR / 'tables'
RPT_DIR      = RESULTS_DIR / 'reports'

for d in (FIG_DIR, TBL_DIR, RPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10,
})
sns.set_style('whitegrid')

print("Environment ready.")
print(f"Project root         : {PROJECT_ROOT}")
print(f"Master seed          : {SEED}")


## 2. Reproduce Task 1 Pipeline (Shared Setup)

The block below is byte-for-byte identical to the corresponding block in Tasks 1 and 2.


In [ ]:
DATASET_PATH = DATA_DIR / 'hotel_bookings.csv'

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}. "
        f"Place hotel_bookings.csv in {DATA_DIR}."
    )

with open(DATASET_PATH, 'rb') as f:
    DATASET_MD5 = hashlib.md5(f.read()).hexdigest()

df_raw = pd.read_csv(DATASET_PATH)
print(f"Dataset MD5      : {DATASET_MD5}")
print(f"Shape (raw)      : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

df = df_raw[df_raw['distribution_channel'] == 'TA/TO'].copy()

df['party_size']    = df['adults'] + df['children'].fillna(0) + df['babies']
df['total_nights']  = df['stays_in_week_nights'] + df['stays_in_weekend_nights']
df = df[df['party_size'] > 0].copy()
df['country']       = df['country'].fillna('PRT')
df['children']      = df['children'].fillna(0)
df['party_size']    = df['adults'] + df['children'] + df['babies']
df['total_nights']  = df['stays_in_week_nights'] + df['stays_in_weekend_nights']
df['weekend_share'] = np.where(df['total_nights'] > 0,
                               df['stays_in_weekend_nights'] / df['total_nights'], 0.0)

df['previous_cancellations']         = df['previous_cancellations'].clip(upper=df['previous_cancellations'].quantile(0.99))
df['previous_bookings_not_canceled'] = df['previous_bookings_not_canceled'].clip(upper=df['previous_bookings_not_canceled'].quantile(0.99))
df['required_car_parking_spaces']    = df['required_car_parking_spaces'].clip(upper=3)
df['total_of_special_requests']      = df['total_of_special_requests'].clip(upper=5)

RARE_THRESHOLD = 0.01
cat_inputs = ['market_segment', 'reserved_room_type', 'meal',
              'customer_type', 'deposit_type', 'country', 'arrival_date_month']
for col in cat_inputs:
    freq = df[col].value_counts(normalize=True)
    rare = freq[freq < RARE_THRESHOLD].index.tolist()
    if rare:
        df[col] = df[col].where(~df[col].isin(rare), other='Other')

print(f"Shape (TA/TO clean) : {df.shape[0]:,} rows")


In [ ]:
# R0 preprocessing
NUM_FEATURES_NUM = ['lead_time',
                    'previous_cancellations', 'previous_bookings_not_canceled',
                    'total_nights', 'weekend_share', 'party_size',
                    'required_car_parking_spaces', 'total_of_special_requests']
NUM_FEATURES_BIN = ['is_repeated_guest']
CAT_FEATURES     = ['market_segment', 'reserved_room_type', 'meal',
                    'customer_type', 'deposit_type', 'country', 'arrival_date_month']

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
numeric_bin_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor_R0 = ColumnTransformer(
    transformers=[
        ('num',     numeric_pipe,     NUM_FEATURES_NUM),
        ('num_bin', numeric_bin_pipe, NUM_FEATURES_BIN),
        ('cat',     cat_pipe,         CAT_FEATURES),
    ],
    remainder='drop',
)

X_R0 = preprocessor_R0.fit_transform(df).astype(np.float64)
ohe = preprocessor_R0.named_transformers_['cat']['onehot']
cat_names = ohe.get_feature_names_out(CAT_FEATURES).tolist()
feature_names_R0 = NUM_FEATURES_NUM + NUM_FEATURES_BIN + cat_names

REPRESENTATION_ID = 'R0-standard-noLog-noADR-marketSegment-countryRare1pct'

print(f"R0 matrix shape       : {X_R0.shape[0]:,} rows x {X_R0.shape[1]} columns")
print(f"representation_id     : {REPRESENTATION_ID}")


In [ ]:
def compute_internal_indices(X, labels, subsample_n=SUBSAMPLE_N, seed=SEED):
    """Internal indices computed in the same Euclidean representation as clustering."""
    n = len(labels)
    if n > subsample_n:
        rng = np.random.default_rng(seed)
        idx = rng.choice(n, size=subsample_n, replace=False)
        X_s, l_s = X[idx], labels[idx]
    else:
        X_s, l_s = X, labels

    if len(np.unique(l_s)) < 2:
        return {'silhouette': np.nan, 'calinski_harabasz': np.nan, 'davies_bouldin': np.nan}

    sil = silhouette_score(X_s, l_s, metric='euclidean')
    ch  = calinski_harabasz_score(X, labels)
    db  = davies_bouldin_score(X, labels)
    return {'silhouette': sil, 'calinski_harabasz': ch, 'davies_bouldin': db}

print("compute_internal_indices() ready.")


## 3. Fuzzy c-Means: Algorithm Specification

### 3.1 Mathematical Formulation (Ppt T5)

Given the encoded matrix $X = \{\mathbf{x}_1, \mathbf{x}_2, \ldots, \mathbf{x}_n\}$,
$\mathbf{x}_i \in \mathbb{R}^d$, FCM minimises the weighted sum of squared distances:

$$
\min_{(W,\,C)} \left\{ J_m(W, C) = \sum_{k=1}^{K} \sum_{i=1}^{n} w_{ik}^{m} \, D_{ik}^2 \right\}, \qquad m > 1
$$

where $D_{ik}^2 = \|\mathbf{x}_i - \mathbf{c}_k\|^2$ and the **membership matrix**
$W = [w_{ik}] \in \mathbb{R}^{n \times K}$ satisfies:

$$
0 \le w_{ik} \le 1 \quad \forall\, i,k; \qquad \sum_{k=1}^{K} w_{ik} = 1 \quad \forall\, i; \qquad 0 < \sum_{i=1}^{n} w_{ik} < n \quad \forall\, k.
$$

**Output 2:** $K$ cluster prototypes $\mathbf{c}_1, \ldots, \mathbf{c}_K \in \mathbb{R}^d$, stored row-wise in
$C \in \mathbb{R}^{K \times d}$.

The parameter $m > 1$ is the **degree of fuzzification** (weighting exponent). $m = 1$
gives a crisp partition; $m = 2$ is the typical default.

### 3.2 Alternating Optimization (AO): Update Rules (Ppt T5)

Starting from an initial prototype matrix $C^{(0)} \in \mathbb{R}^{K \times d}$,
the algorithm repeats two closed-form updates derived from first-order optimality conditions:

**Membership update**: for $i = 1, \ldots, n;\; k = 1, \ldots, K$:

$$
w_{ik}^{(t)} \leftarrow \frac{\left(\dfrac{1}{D_{ik}^{(t-1)}}\right)^{\!\frac{2}{m-1}}}{\displaystyle\sum_{q=1}^{K} \left(\dfrac{1}{D_{iq}^{(t-1)}}\right)^{\!\frac{2}{m-1}}}
$$

**Prototype update**: for $k = 1, \ldots, K$:

$$
\mathbf{c}_k^{(t)} \leftarrow \frac{\displaystyle\sum_{i=1}^{n} \left(w_{ik}^{(t)}\right)^m \mathbf{x}_i}{\displaystyle\sum_{i=1}^{n} \left(w_{ik}^{(t)}\right)^m}
$$

### 3.3 Termination Criterion (PPT 5)

The loop terminates when:

$$
\left|\mathbf{c}_k^{(t)} - \mathbf{c}_k^{(t-1)}\right| \le \varepsilon \quad \text{for all } k = 1,\ldots,K,
\qquad \text{or} \qquad t = T.
$$

Typical values: $\varepsilon = 0.001$, $T = 100$.

### 3.4 Initialisation

Prototypes are initialised by a **K-Means warm start** (`init='k-means++'`, `n_init=5`),
keeping the FCM solution directly comparable to the K-Means partition at the same `K`.


In [ ]:
def fuzzy_cmeans(X, K, m=2.0, tol=1e-3, max_iter=100,
                 init_prototypes=None, seed=SEED, verbose=False):
    """Fuzzy c-Means – Alternating Optimization (AO) scheme.

    Notation follows T5 slides (Nascimento, 2025/2026):
      W = [w_ik] in R^{n x K}  – membership matrix
      C in R^{K x d}            – prototype matrix (row k = c_k)
      J_m(W, C)                 – FCM objective
      D_ik = ||x_i - c_k||      – Euclidean distance

    Termination: |c_k^{(t)} - c_k^{(t-1)}| <= tol for all k, or t = T.

    Parameters
    ----------
    X   : ndarray (n, d)
    K   : int, number of clusters (1 < K < n)
    m   : float > 1, degree of fuzzification (typical: 2.0)
    tol : float > 0, termination threshold epsilon (typical: 0.001)
    max_iter : int, maximum iterations T (typical: 100)

    Returns
    -------
    dict with keys W, C, J_m, n_iter, stop_reason, time_s
    """
    X = np.ascontiguousarray(X, dtype=np.float64)
    n, d = X.shape
    assert K >= 2 and m > 1.0, "K must be >= 2 and m must be > 1"
    exp = 2.0 / (m - 1.0)          # exponent for membership update

    # Initialisation C^{(0)}
    if init_prototypes is None:
        km0 = KMeans(n_clusters=K, init='k-means++', n_init=5, max_iter=300,
                     random_state=seed).fit(X)
        C = km0.cluster_centers_.astype(np.float64).copy()
    else:
        C = np.asarray(init_prototypes, dtype=np.float64).copy()
        assert C.shape == (K, d)

    def _update_W(X, C):
        """Membership update: w_ik = (1/D_ik)^p / sum_q (1/D_iq)^p, p=2/(m-1)."""
        D2 = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)   # (n, K)
        zero_mask = D2 == 0
        if zero_mask.any():
            W = np.zeros_like(D2)
            for i in np.where(zero_mask.any(axis=1))[0]:
                j = np.argmin(D2[i])
                W[i, j] = 1.0
            non_zero = ~zero_mask.any(axis=1)
            if non_zero.any():
                D = np.sqrt(D2[non_zero])
                ratios = (D[:, :, None] / D[:, None, :]) ** exp
                W[non_zero] = 1.0 / ratios.sum(axis=2)
            return W
        D = np.sqrt(D2)
        ratios = (D[:, :, None] / D[:, None, :]) ** exp
        return 1.0 / ratios.sum(axis=2)

    def _update_C(X, W, m):
        """Prototype update: c_k = sum_i w_ik^m x_i / sum_i w_ik^m."""
        Wm = W ** m
        return (Wm.T @ X) / np.maximum(Wm.sum(axis=0)[:, None], 1e-12)

    def _objective(X, W, C, m):
        """J_m(W, C) = sum_k sum_i w_ik^m * D_ik^2."""
        D2 = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
        return float(((W ** m) * D2).sum())

    t0   = time.time()
    W    = _update_W(X, C)
    stop = 'max_iter'

    for it in range(max_iter):
        C_new = _update_C(X, W, m)
        W_new = _update_W(X, C_new)

        # Termination: |c_k^{(t)} - c_k^{(t-1)}| <= eps  for all k
        centroid_change = np.max(np.abs(C_new - C))

        if verbose:
            J_now = _objective(X, W_new, C_new, m)
            print(f"  iter {it+1:3d}: J_m = {J_now:.4e} | max|dc_k| = {centroid_change:.2e}")

        C = C_new
        W = W_new

        if centroid_change <= tol:
            stop = f'centroid_stability (max|c_k^t - c_k^{{t-1}}| <= {tol:.0e})'
            break

    elapsed = time.time() - t0
    J_final = _objective(X, W, C, m)
    return {'W': W, 'C': C, 'J_m': J_final, 'n_iter': it + 1,
            'stop_reason': stop, 'time_s': elapsed}


def fuzzy_validity_indices(X, W, C, m=2.0):
    """Membership-based internal validity indices (T5, slides 48–50).

    PC(K)  = (1/n) sum_i sum_k w_ik^2              [maximise]
    CE(K)  = -(1/n) sum_i sum_k w_ik * log(w_ik)   [minimise]
    XB(K)  = J_m(W,C) / (n * min_{k!=q} ||c_k - c_q||^2)  [minimise]
    """
    n = W.shape[0]
    K = C.shape[0]

    PC = float((W ** 2).sum() / n)

    with np.errstate(divide='ignore', invalid='ignore'):
        log_W = np.where(W > 0, np.log(W), 0.0)
    CE = float(-(W * log_W).sum() / n)

    D2 = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    J_m = float(((W ** m) * D2).sum())

    min_sep2 = np.inf
    for j in range(K):
        for q in range(K):
            if j != q:
                sep2 = float(((C[j] - C[q]) ** 2).sum())
                if sep2 < min_sep2:
                    min_sep2 = sep2
    XB = J_m / (n * min_sep2)

    return {'PC': PC, 'CE': CE, 'XB': XB}


print("fuzzy_cmeans() and fuzzy_validity_indices() ready.")


## 4. Calibrating the Fuzziness Exponent `m` for R0

### 4.1 Why this step is necessary

The slides (T5, slide 37) note that `m = 2.0` is the typical default. However, in a
**high-dimensional** representation such as R0 (`d = 57`), the default `m = 2.0` can
produce a degenerate solution where all memberships collapse to `1/K` (uniform) and all
prototypes drift to the global mean. We verify this empirically and select an operating `m`
that keeps the partition informative.

### 4.2 Empirical calibration sweep

We sweep `m` over a predeclared grid and record:

- **Partition Coefficient PC(K):** crisp if PC = 1, uniform if PC = 1/K.
- **Mean maximum membership:** close to 1 means crisp, close to 1/K means uniform.
- **Minimum inter-prototype distance** after convergence: 0 means prototypes collapsed.
- **ARI vs the K-Means partition at `K = 4`**: soft–hard agreement.

The aim is to locate the transition between "FCM informative" and "FCM degenerate",
and to operate just inside the informative side.


In [ ]:
# K-Means baseline at K = 4 (used as reference partition and for FCM initialisation)
km_baseline = KMeans(n_clusters=4, init='k-means++', n_init=5,
                     max_iter=300, random_state=SEED).fit(X_R0)
km_labels   = km_baseline.labels_
C_init      = km_baseline.cluster_centers_.astype(np.float64)   # C^{(0)}

# Calibration grid
M_CALIBRATION = [1.05, 1.08, 1.10, 1.13, 1.15, 1.20, 1.50, 2.00]
d_R0          = X_R0.shape[1]

print(f"Representation dimension d : {d_R0}")
print()
print("Calibration sweep:")
print(f"{'m':>6} | {'PC':>7} | {'mean_max_w':>10} | {'min_||c||':>10} | "
      f"{'iters':>5} | {'ARI(KM)':>8} | {'verdict':>20}")
print("-" * 90)

calibration_rows = []
for m_val in M_CALIBRATION:
    res = fuzzy_cmeans(X_R0, K=4, m=m_val, init_prototypes=C_init,
                       seed=SEED, verbose=False)
    W_cal, C_cal = res['W'], res['C']

    PC       = (W_cal ** 2).sum() / W_cal.shape[0]
    mean_max = W_cal.max(axis=1).mean()
    proto_dists = [np.linalg.norm(C_cal[i] - C_cal[j])
                   for i in range(4) for j in range(i+1, 4)]
    min_cd   = min(proto_dists)
    labels_m = W_cal.argmax(axis=1)
    ari_m    = adjusted_rand_score(km_labels, labels_m)

    if mean_max < 0.30:
        verdict = 'DEGENERATE'
    elif mean_max < 0.50:
        verdict = 'near-uniform'
    elif mean_max < 0.70:
        verdict = 'moderately soft'
    elif mean_max < 0.90:
        verdict = 'soft, informative'
    else:
        verdict = 'near-crisp'

    calibration_rows.append({
        'm': m_val, 'PC': PC, 'mean_max_w': mean_max,
        'min_proto_dist': min_cd, 'n_iter': res['n_iter'],
        'ARI_vs_KMeans': ari_m, 'verdict': verdict,
    })
    print(f"{m_val:>6.2f} | {PC:>7.4f} | {mean_max:>10.4f} | {min_cd:>10.4f} | "
          f"{res['n_iter']:>5} | {ari_m:>8.4f} | {verdict:>20}")

calibration_df = pd.DataFrame(calibration_rows)
calibration_df.to_csv(TBL_DIR / 'fcm_m_calibration.csv', index=False)


In [ ]:
# Figure E3-1: calibration curve
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

m_vals = calibration_df['m'].values

ax = axes[0]
ax.plot(m_vals, calibration_df['PC'], 'o-', color='#1565C0', linewidth=2, markersize=8)
ax.axhline(1.0/4, color='red', linestyle=':', alpha=0.6, label='uniform limit (1/K = 0.25)')
ax.set_xlabel('Fuzzification exponent m')
ax.set_ylabel('Partition Coefficient PC(K)')
ax.set_title('PC vs m  (higher = crisper)')
ax.legend(fontsize=8, loc='upper right')
ax.set_xscale('log')

ax = axes[1]
ax.plot(m_vals, calibration_df['mean_max_w'], 'o-', color='#2E7D32', linewidth=2, markersize=8)
ax.axhline(1.0/4, color='red', linestyle=':', alpha=0.6, label='uniform limit')
ax.set_xlabel('Fuzzification exponent m')
ax.set_ylabel('Mean max membership  max_k w_ik')
ax.set_title('Mean argmax-membership vs m')
ax.legend(fontsize=8, loc='upper right')
ax.set_xscale('log')

ax = axes[2]
ax.plot(m_vals, calibration_df['min_proto_dist'], 'o-', color='#E65100', linewidth=2, markersize=8)
ax.set_xlabel('Fuzzification exponent m')
ax.set_ylabel('min ||c_k - c_q||')
ax.set_title('Minimum inter-prototype distance vs m')
ax.set_xscale('log')

fig.suptitle(f'Figure E3-1 – Calibrating m on R0 (d = {d_R0})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'figE3_1_m_calibration.png')
plt.show()

print("Reading the figure.")
print(f"- For small m, PC > 1/K and prototypes are well separated: FCM is informative.")
print(f"- For large m (e.g. m = 2.0 on d = {d_R0}), the algorithm collapses to uniform")
print(f"  memberships and the prototypes merge at the global mean.")


### 4.3 Operating choice of `m`

We pick `m = 1.13` as the operating fuzzification exponent. At this value memberships
are visibly soft (PC < 1, mean max < 1) but the prototypes remain well separated and
the FCM hard partition (argmax of W) is highly consistent with the K-Means partition
at the same `K = 4`. Values `m ∈ {1.05, 1.10, 1.13}` are all defensible; we take
`m = 1.13` for the headline analysis and report the full sensitivity in the
experiments log.


## 5. Main Fit – FCM at `K = 4`, `m = 1.13`


In [ ]:
K           = 4
M_OPERATING = 1.13

print(f"Main FCM fit: K = {K}, m = {M_OPERATING}  on X_R0 ({X_R0.shape[0]:,} x {X_R0.shape[1]})")
print()

fcm = fuzzy_cmeans(X_R0, K=K, m=M_OPERATING, init_prototypes=C_init,
                   seed=SEED, verbose=False)

print(f"Converged after {fcm['n_iter']} iterations  ({fcm['stop_reason']})")
print(f"Final objective J_m : {fcm['J_m']:.2f}")
print(f"Runtime             : {fcm['time_s']:.2f} s")

W_main    = fcm['W']          # membership matrix  W = [w_ik] in R^{n x K}
C_main    = fcm['C']          # prototype matrix   C in R^{K x d}
fcm_labels = np.argmax(W_main, axis=1)    # crisp labels z_i = argmax_k w_ik

# Cluster sizes (from hard assignment)
sizes = pd.Series(fcm_labels).value_counts().sort_index()
print()
print(f"Cluster sizes (argmax of W,  K = {K}):")
for k_id, n_k in sizes.items():
    print(f"  Cluster {k_id}: {n_k:>7,}  ({n_k/len(fcm_labels)*100:.1f}%)")


## 6. Internal Validity Indices

Two families of indices, computed in the same R0 Euclidean space used for clustering.


In [ ]:
crisp_idx = compute_internal_indices(X_R0, fcm_labels)
fuzzy_idx = fuzzy_validity_indices(X_R0, W_main, C_main, m=M_OPERATING)

print(f"FCM internal validity at K = {K}, m = {M_OPERATING}")
print("=" * 60)
print()
print("Crisp indices (on argmax labels z_i = argmax_k w_ik):")
print(f"  Silhouette        : {crisp_idx['silhouette']:.4f}   (higher = better)")
print(f"  Calinski-Harabasz : {crisp_idx['calinski_harabasz']:,.0f}   (higher = better)")
print(f"  Davies-Bouldin    : {crisp_idx['davies_bouldin']:.4f}   (lower = better)")
print()
print("Membership-based fuzzy indices (on W):")
print(f"  PC(K)  = (1/n) Σ_i Σ_k w_ik^2       : {fuzzy_idx['PC']:.4f}   (higher = crisper, max = 1)")
print(f"  CE(K)  = -(1/n) Σ_i Σ_k w_ik log w_ik: {fuzzy_idx['CE']:.4f}   (lower = crisper, min = 0)")
print(f"  XB(K)  = J_m / (n * min ||c_k-c_q||^2): {fuzzy_idx['XB']:.6f}   (lower = better)")
print()
print(f"Comparison – K-Means at K = {K} on R0:")
km_idx = compute_internal_indices(X_R0, km_labels)
print(f"  K-Means Silhouette        : {km_idx['silhouette']:.4f}")
print(f"  K-Means Calinski-Harabasz : {km_idx['calinski_harabasz']:,.0f}")
print(f"  K-Means Davies-Bouldin    : {km_idx['davies_bouldin']:.4f}")


## 7. Membership Confidence by Cluster

The central question: *which clusters are crisp (high w_ik for the dominant k)
and which are soft (w_ik spread across multiple clusters)?*

The hypothesis is that the small behavioural minorities are sharply identified
(max_k w_ik ≈ 1) while the large mid-clusters show lower confidence.


In [ ]:
max_mem = W_main.max(axis=1)      # max_k w_ik for each observation i

df_mem = pd.DataFrame({'cluster': fcm_labels, 'max_membership': max_mem})
mem_summary = (
    df_mem.groupby('cluster')['max_membership']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .round(3)
)
mem_summary['pct_confident'] = (
    df_mem.groupby('cluster')['max_membership']
    .apply(lambda s: (s >= 0.80).mean() * 100)
    .round(1)
)
mem_summary['pct_ambiguous'] = (
    df_mem.groupby('cluster')['max_membership']
    .apply(lambda s: (s < 0.60).mean() * 100)
    .round(1)
)

print(f"Membership confidence by cluster  (K = {K}, m = {M_OPERATING})")
print("=" * 90)
print(mem_summary.to_string())
print()
print("Legend.")
print("  pct_confident : share of bookings with max_k w_ik >= 0.80")
print("  pct_ambiguous : share of bookings with max_k w_ik <  0.60")

mem_summary.to_csv(TBL_DIR / 'fcm_membership_summary.csv')


In [ ]:
# Figure E3-2: histogram of max membership, facetted by dominant cluster
fig, axes = plt.subplots(1, K, figsize=(15, 3.6), sharey=True)
colors = sns.color_palette('Set2', K)

for k_id in range(K):
    ax = axes[k_id]
    sub = df_mem[df_mem['cluster'] == k_id]['max_membership']
    ax.hist(sub, bins=30, range=(1.0/K, 1.0),
            color=colors[k_id], alpha=0.85, edgecolor='white')
    ax.axvline(0.80, color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(1.0/K, color='red',   linestyle=':',  alpha=0.5, linewidth=1)
    pct_conf = (sub >= 0.80).mean() * 100
    ax.set_title(f'Cluster {k_id}  (n = {len(sub):,})\n{pct_conf:.1f}% confident')
    ax.set_xlabel('max_k w_ik')
    ax.set_xlim(1.0/K - 0.02, 1.02)
    if k_id == 0:
        ax.set_ylabel('Frequency')

fig.suptitle(f'Figure E3-2 – Membership confidence by cluster  (K = {K}, m = {M_OPERATING})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'figE3_2_max_membership_by_cluster.png')
plt.show()

print(f"Dashed line = 0.80 (confidence threshold).")
print(f"Dotted line = {1.0/K:.2f} (uniform-membership lower bound 1/K).")


### 7.1 Where do the ambiguous bookings sit?

For each ambiguous booking (max_k w_ik < 0.60) we record the **second-choice cluster**.
If the diffuse-boundary hypothesis is correct, the dominant transition should be
between the two large engagement clusters.


In [ ]:
mask_amb = max_mem < 0.60
n_amb    = int(mask_amb.sum())

if n_amb > 0:
    sorted_clusters = np.argsort(-W_main, axis=1)
    first_choice    = sorted_clusters[:, 0]
    second_choice   = sorted_clusters[:, 1]

    transition_tbl = pd.crosstab(
        pd.Series(first_choice[mask_amb], name='1st choice (argmax)'),
        pd.Series(second_choice[mask_amb], name='2nd choice'),
    )
    print(f"Ambiguous bookings (max_k w_ik < 0.60): {n_amb:,} "
          f"({n_amb / len(max_mem) * 100:.1f}% of the sub-population)")
    print()
    print("Cross-tab: 1st-choice cluster x 2nd-choice cluster (counts):")
    print(transition_tbl.to_string())
    transition_tbl.to_csv(TBL_DIR / 'fcm_ambiguous_transitions.csv')

    transition_pct = transition_tbl.div(transition_tbl.sum(axis=1), axis=0).mul(100).round(1)
    print()
    print("Same cross-tab normalised by row (%):")
    print(transition_pct.to_string())
else:
    print("No ambiguous bookings under the 0.60 threshold.")


## 8. Agreement with K-Means at `K = 4`


In [ ]:
ari_km_fcm = adjusted_rand_score(km_labels, fcm_labels)

print(f"K-Means cluster sizes:")
for k_id, n_k in pd.Series(km_labels).value_counts().sort_index().items():
    print(f"  Cluster {k_id}: {n_k:>7,}  ({n_k/len(km_labels)*100:.1f}%)")

print()
print(f"FCM (argmax W) cluster sizes:")
for k_id, n_k in pd.Series(fcm_labels).value_counts().sort_index().items():
    print(f"  Cluster {k_id}: {n_k:>7,}  ({n_k/len(fcm_labels)*100:.1f}%)")

print()
print(f"ARI(K-Means(K=4), FCM(K={K}, m={M_OPERATING})) on X_R0 : {ari_km_fcm:.4f}")
print()
if ari_km_fcm >= 0.90:
    print("Very high agreement: the FCM hard view (argmax W) essentially reproduces the")
    print("  K-Means partition. The added value of FCM is the membership matrix W.")
elif ari_km_fcm >= 0.75:
    print("High agreement: the FCM hard view and the K-Means partition describe the same")
    print("  core structure, with minor reassignments around diffuse boundaries.")
else:
    print("Moderate agreement: the FCM and K-Means partitions diverge meaningfully –")
    print("  this is itself informative about partition stability.")


## 9. Cluster Profiling (FCM hard labels, original units)


In [ ]:
df_fcm = df.copy()
df_fcm['cluster_fcm']    = fcm_labels
df_fcm['max_membership'] = max_mem

ALL_NUM = NUM_FEATURES_NUM + NUM_FEATURES_BIN

profile_num_fcm = (
    df_fcm.groupby('cluster_fcm')[ALL_NUM]
    .agg(['mean', 'median', 'std'])
    .round(2)
)
print("FCM CLUSTER PROFILES – Numerical inputs (original units)")
print("=" * 90)
print(profile_num_fcm.to_string())
profile_num_fcm.to_csv(TBL_DIR / 'fcm_profile_numerical.csv')


In [ ]:
print("FCM CLUSTER PROFILES – Categorical inputs (modal value)")
print("=" * 80)
modes_fcm = {}
for feat in CAT_FEATURES:
    m_modes = df_fcm.groupby('cluster_fcm')[feat].agg(lambda s: s.value_counts().index[0])
    modes_fcm[feat] = m_modes
    print(f"  {feat:25s}: {dict(m_modes)}")
pd.DataFrame(modes_fcm).to_csv(TBL_DIR / 'fcm_profile_categorical_mode.csv')


In [ ]:
df_fcm['is_canceled']     = df['is_canceled'].values
df_fcm['booking_changes'] = df['booking_changes'].values
df_fcm['adr']             = df['adr'].values
df_fcm['hotel']           = df['hotel'].values

posthoc_fcm = df_fcm.groupby('cluster_fcm').agg(
    cluster_size         = ('is_canceled', 'count'),
    cancellation_rate    = ('is_canceled', lambda s: s.mean() * 100),
    mean_booking_changes = ('booking_changes', 'mean'),
    mean_max_membership  = ('max_membership', 'mean'),
    median_adr           = ('adr', lambda s: s[s > 0].median()),
).round(3)
posthoc_fcm['cancellation_rate'] = posthoc_fcm['cancellation_rate'].round(1)

print("POST-HOC PROFILING – FCM clusters")
print("=" * 90)
print(posthoc_fcm.to_string())
posthoc_fcm.to_csv(TBL_DIR / 'fcm_profile_posthoc.csv')


In [ ]:
# Figure E3-3 – Radar of FCM cluster profiles (prototypes C_main, normalised)
profile_means_fcm = df_fcm.groupby('cluster_fcm')[ALL_NUM].mean()
profile_norm_fcm  = (profile_means_fcm - profile_means_fcm.min()) / \
                    (profile_means_fcm.max() - profile_means_fcm.min() + 1e-9)

categories = ALL_NUM
N      = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))
colors  = sns.color_palette('Set2', K)

for k_id in range(K):
    vals = profile_norm_fcm.loc[k_id].tolist()
    vals += [vals[0]]
    ax.plot(angles, vals, 'o-', linewidth=2, label=f'Cluster {k_id}', color=colors[k_id])
    ax.fill(angles, vals, alpha=0.12, color=colors[k_id])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace('_', '\n') for f in categories], size=9)
ax.set_yticklabels([])
ax.set_title(f'Figure E3-3 – FCM Cluster Profiles (K = {K}, m = {M_OPERATING})',
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.32, 1.10))
plt.tight_layout()
plt.savefig(FIG_DIR / 'figE3_3_radar_fcm.png')
plt.show()


## 10. Experiment Logging


In [ ]:
exp_rows_e3 = []

# Main fit
exp_rows_e3.append({
    'date'              : pd.Timestamp.now().strftime('%Y-%m-%d'),
    'run_id'            : f'fcm-K{K}-m{M_OPERATING}-seed{SEED}',
    'representation_id' : REPRESENTATION_ID,
    'method'            : 'FuzzyCMeans',
    'parameters'        : f'K={K},m={M_OPERATING},init=KMeans(k-means++,n_init=5),'
                          f'tol=1e-3,max_iter=100',
    'seed'              : SEED,
    'sample_rule'       : 'full TA/TO sub-population',
    'k'                 : K,
    'silhouette'        : crisp_idx['silhouette'],
    'calinski_harabasz' : crisp_idx['calinski_harabasz'],
    'davies_bouldin'    : crisp_idx['davies_bouldin'],
    'time_s'            : fcm['time_s'],
    'notes'             : f"E3 main fit (calibrated m). PC={fuzzy_idx['PC']:.3f}, "
                          f"CE={fuzzy_idx['CE']:.3f}, XB={fuzzy_idx['XB']:.6f}, "
                          f"ARI(K-Means)={ari_km_fcm:.4f}",
})

# m calibration sweep
for r in calibration_rows:
    exp_rows_e3.append({
        'date'              : pd.Timestamp.now().strftime('%Y-%m-%d'),
        'run_id'            : f'fcm-K{K}-m{r["m"]}-calib-seed{SEED}',
        'representation_id' : REPRESENTATION_ID,
        'method'            : 'FuzzyCMeans_m_calibration',
        'parameters'        : f'K={K},m={r["m"]},init=KMeans(k-means++,n_init=5)',
        'seed'              : SEED,
        'sample_rule'       : 'full TA/TO sub-population',
        'k'                 : K,
        'silhouette'        : np.nan,
        'calinski_harabasz' : np.nan,
        'davies_bouldin'    : np.nan,
        'time_s'            : None,
        'notes'             : f"E3 m-calibration. PC={r['PC']:.4f}, "
                              f"mean_max={r['mean_max_w']:.4f}, "
                              f"min_proto_dist={r['min_proto_dist']:.4f}, "
                              f"n_iter={r['n_iter']}, ARI_KM={r['ARI_vs_KMeans']:.4f}, "
                              f"verdict={r['verdict']}",
    })

df_exp_e3 = pd.DataFrame(exp_rows_e3)
exp_path  = RPT_DIR / 'experiments.csv'
mode      = 'a' if exp_path.exists() else 'w'
header    = (mode == 'w')
df_exp_e3.to_csv(exp_path, mode=mode, header=header, index=False)
df_exp_e3.to_csv(TBL_DIR / 'e3_results.csv', index=False)

print(f"Wrote {len(df_exp_e3)} rows to experiments.csv (mode={mode}).")
print(f"Saved standalone table to tables/e3_results.csv")
print()
print(df_exp_e3[['run_id', 'method', 'k', 'silhouette', 'notes']].to_string(index=False))


## 11. Conclusions

1. **The fuzzification exponent must be calibrated for R0.** The typical default
   `m = 2.0` (T5, slide 37) produces a degenerate solution on this 57-dimensional
   representation: memberships collapse to `1/K` and prototypes merge at the global mean.
   The empirical calibration sweep identifies `m = 1.13` as a value that keeps the
   partition informative. *This is a methodological finding worth discussing in the report.*

2. **At the operating choice `m = 1.13`**, the FCM hard view (argmax of `W`) reproduces
   the K-Means partition at `K = 4` with high ARI. The added value of FCM over K-Means
   is the full membership matrix `W = [w_ik]`, which provides a continuous confidence
   layer on top of the crisp assignment.

3. **The fuzzy validity indices PC(K), CE(K), and XB(K)** (T5, slides 49–50) quantify
   partition quality in terms of the membership matrix `W` and the prototype matrix `C`.
   The Xie–Beni index is particularly informative because it uses both `W` and `C`.

4. **Membership confidence is cluster-dependent.** Small behavioural minorities are
   sharply identified (max_k w_ik ≈ 1). The large mid-clusters have lower argmax
   memberships, with a non-trivial share of bookings below the 0.60 ambiguity threshold.
   The cross-tab of 1st vs 2nd choice confirms that ambiguous bookings hesitate
   predominantly between the two large clusters.

5. **This is consistent with the Task 1 bootstrap stability finding** (mean ARI = 0.644,
   min = 0.139). The moderate stability reflects a real diffuse boundary between the two
   large engagement clusters – precisely what FCM allows us to articulate quantitatively
   through the membership matrix W.
